In [3]:
import pandas as pd
import json
from fractions import Fraction
import re

# Helper to parse quantity, unit, and ingredient
def parse_quantity_and_ingredient(text):
    text = text.lower().strip()
    pattern = r'^([\d\s\/\.]+)\s*(\w+)?\s*(.*)'
    match = re.match(pattern, text)

    if match:
        quantity_str = match.group(1).strip()
        unit = match.group(2)
        ingredient_name = match.group(3).strip()

        try:
            if ' ' in quantity_str:
                parts = quantity_str.split()
                quantity = float(parts[0]) + float(Fraction(parts[1]))
            else:
                quantity = float(Fraction(quantity_str))
        except:
            quantity = 1.0

        return quantity, unit, ingredient_name
    else:
        return 1.0, None, text

# Load the flavor profile data (ingredient-to-flavor mapping)
flavor_df = pd.read_csv('flavour_scaled_cleaned.csv')

# Map flavor profiles to a dictionary for faster lookup
flavor_mapping = {
    row['ingredient_name']: {
        'salty': row['salty'],
        'spicy': row['spicy'],
        'sweet': row['sweet'],
        'sour': row['sour'],
        'bitter': row['bitter'],
        'umami': row['umami']
    }
    for index, row in flavor_df.iterrows()
}

# Load the dishes data
with open('final_cleaned_tarla_dalal.json', 'r', encoding='utf-8') as file:
    dishes_data = json.load(file)

# Prepare the new structure for the output CSV
output_data = []

# Iterate over each dish and map flavor profiles
for dish in dishes_data:
    dish_name = dish['name']
    ingredients = dish['ingredients']
    instructions = ' '.join(dish['instructions'])
    region = dish['region']

    # Initialize the flavor levels to 0
    salty, spicy, sweet, sour, bitter, umami = 0, 0, 0, 0, 0, 0

    # Iterate over each ingredient
    for ingredient in ingredients:
        quantity, unit, ingredient_name = parse_quantity_and_ingredient(ingredient)

        # Check if the clean ingredient exists in the flavor mapping
        if ingredient_name in flavor_mapping:
            salty += flavor_mapping[ingredient_name]['salty'] * quantity
            spicy += flavor_mapping[ingredient_name]['spicy'] * quantity
            sweet += flavor_mapping[ingredient_name]['sweet'] * quantity
            sour += flavor_mapping[ingredient_name]['sour'] * quantity
            bitter += flavor_mapping[ingredient_name]['bitter'] * quantity
            umami += flavor_mapping[ingredient_name]['umami'] * quantity

    # Append the mapped data
    output_data.append({
        'name': dish_name,
        'ingredients': ', '.join(ingredients),
        'salty': salty,
        'spicy': spicy,
        'sweet': sweet,
        'sour': sour,
        'bitter': bitter,
        'umami': umami,
        'instructions': instructions,
        'region': region
    })

# Convert to DataFrame and save
output_df = pd.DataFrame(output_data)
output_df.to_csv('mapped_dishes_with_flavors_and_region_scaled.csv', index=False)

print("✅ Flavour mapping with quantity scaling complete! Results saved in 'mapped_dishes_with_flavors_and_region_scaled.csv'.")


✅ Flavour mapping with quantity scaling complete! Results saved in 'mapped_dishes_with_flavors_and_region_scaled.csv'.
